In [3]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
from datetime import datetime, timedelta, UTC, timezone
from zoneinfo import ZoneInfo
from google.cloud import bigquery
import os, re
import time, random
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import sys
sys.path.append("../src")

from config import GCP_PROJECT_ID, RAW_FLIGHTS_TABLE_ID, FLIGHT_API_URL, HEADERS

In [5]:
def date_range(start: str, end: str):
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")

    while start_dt <= end_dt:
        yield start_dt.strftime("%Y%m%d")
        start_dt += timedelta(days=1)

In [6]:
_TIME_HHMM = re.compile(r"^\d{2}:\d{2}$")
KST = ZoneInfo("Asia/Seoul")
UTC = timezone.utc  # 표준 UTC tzinfo

def parse_dt_utc(ymd: str, time_str: str | None):
    """
    AirPortal time_str ('HH:MM') + ymd ('YYYYMMDD')를
    KST로 해석한 뒤 UTC-aware datetime으로 반환.
    - None/형식 불일치: None
    - '24:00': 다음날 00:00으로 처리
    """
    if time_str is None:
        return None

    s = str(time_str).strip()
    if not _TIME_HHMM.match(s):
        return None

    # 24:00 처리
    base_date = datetime.strptime(ymd, "%Y%m%d")
    if s == "24:00":
        base_date = base_date + timedelta(days=1)
        s = "00:00"

    naive = datetime.strptime(f"{base_date.strftime('%Y%m%d')} {s}", "%Y%m%d %H:%M")
    kst_dt = naive.replace(tzinfo=KST)            # KST로 의미 부여
    return kst_dt.astimezone(UTC)                 # UTC로 변환 (aware)

In [7]:
def build_session(
    total_retries: int = 5,
    backoff_factor: float = 1.0,
    status_forcelist: tuple[int, ...] = (429, 500, 502, 503, 504),
    timeout_sec: int = 20,
) -> requests.Session:
    """
    재시도 + 지수 백오프 세션 생성.
    - 429/5xx에 대해 자동 재시도
    - 연결 재사용으로 백필 속도/안정성 개선
    """
    session = requests.Session()

    retry = Retry(
        total=total_retries,
        connect=total_retries,
        read=total_retries,
        status=total_retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=frozenset(["GET", "POST"]),
        raise_on_status=False, 
        respect_retry_after_header=True,  
    )

    adapter = HTTPAdapter(max_retries=retry, pool_connections=50, pool_maxsize=50)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    session.request_timeout = timeout_sec  
    return session

In [8]:
def fetch_flights(session: requests.Session, ymd: str) -> list[dict]:
    payload = {
        "searchContext": "departure",
        "ymd": ymd,
        "startTime": "0000",
        "endTime": "2400",
        "loadType": "json",
        "depAirport": "RKSI",
    }


    res = session.post(
        FLIGHT_API_URL,
        json=payload,
        headers=HEADERS,
        timeout=getattr(session, "request_timeout", 20),
    )

    # 4xx/5xx 최종 실패 처리
    res.raise_for_status()
    return res.json().get("content", [])

In [9]:
def transform(records: list[dict], ymd: str) -> pd.DataFrame:
    rows = []
    collected_at = datetime.now(UTC)

    for r in records:
        flight_iata = r.get("fpIata")

        row = {
            "flight_key": f"{ymd}_{flight_iata}",

            "airline_icao": r.get("alIcao"),
            "airline_kr": r.get("alKr"),
            "flight_iata": flight_iata,

            "arr_airport_iata": r.get("apIata"),
            "arr_airport_kr": r.get("apKr"),

            "status": r.get("status"),
            "status_remark": r.get("statusRemark"),
            "status_remark_code": r.get("statusRemarkCode"),

            "scheduled_utc_ts": parse_dt_utc(ymd, r.get("schTime")),
            "expected_utc_ts": parse_dt_utc(ymd, r.get("expectedFlightTime")),
            "actual_utc_ts": parse_dt_utc(ymd, r.get("actualFlightTime")),

            "nature": r.get("nature"),
            "ymd": datetime.strptime(ymd, "%Y%m%d").date(),
            "collected_at": collected_at,
        }

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
def upload_to_bq(df: pd.DataFrame):
    client = bigquery.Client(project=GCP_PROJECT_ID)

    job = client.load_table_from_dataframe(
        df,
        RAW_FLIGHTS_TABLE_ID,
        job_config=bigquery.LoadJobConfig(
            write_disposition="WRITE_APPEND"
        )
    )
    job.result()

In [11]:
session = build_session()

session.get(FLIGHT_API_URL, timeout=getattr(session, "request_timeout", 20))

failed_days = []

for ymd in date_range("2023-01-01", "2025-12-31"):
    try:
        records = fetch_flights(session, ymd) 

        if not records:
            print(f"{ymd}: no data")
            time.sleep(random.uniform(0.8, 1.5))
            continue

        df = transform(records, ymd)
        upload_to_bq(df)

        print(f"{ymd}: inserted {len(df)} rows")

    except Exception as e:
        print(f"{ymd}: failed - {e}")
        failed_days.append(ymd)

        time.sleep(random.uniform(3.0, 6.0))

print(f"Failed days: {len(failed_days)}")
if failed_days:
    print(failed_days[:20])  

20230101: inserted 360 rows
20230102: inserted 347 rows
20230103: inserted 331 rows
20230104: inserted 395 rows
20230105: inserted 380 rows
20230106: inserted 375 rows
20230107: inserted 409 rows
20230108: inserted 375 rows
20230109: inserted 359 rows
20230110: inserted 355 rows
20230111: inserted 411 rows
20230112: inserted 394 rows
20230113: inserted 390 rows
20230114: inserted 400 rows
20230115: inserted 381 rows
20230116: inserted 363 rows
20230117: inserted 351 rows
20230118: inserted 412 rows
20230119: inserted 405 rows
20230120: inserted 394 rows
20230121: inserted 400 rows
20230122: inserted 375 rows
20230123: inserted 357 rows
20230124: inserted 338 rows
20230125: inserted 372 rows
20230126: inserted 359 rows
20230127: inserted 370 rows
20230128: inserted 392 rows
20230129: inserted 367 rows
20230130: inserted 349 rows
20230131: inserted 362 rows
20230201: inserted 407 rows
20230202: inserted 399 rows
20230203: inserted 374 rows
20230204: inserted 405 rows
20230205: inserted 3

In [ ]:
# 실패한 날 재시도 
ymd = "20240826"
records = fetch_flights(session, ymd)

if not records:
    print(f"{ymd}: no data")
else:
    rows_ok = []
    rows_bad = 0

    collected_at = datetime.now(UTC)

    for r in records:
        try:
            flight_iata = r.get("fpIata")
            row = {
                "flight_key": f"{ymd}_{flight_iata}",
                "airline_icao": r.get("alIcao"),
                "airline_kr": r.get("alKr"),
                "flight_iata": flight_iata,
                "arr_airport_iata": r.get("apIata"),
                "arr_airport_kr": r.get("apKr"),
                "status": r.get("status"),
                "status_remark": r.get("statusRemark"),
                "status_remark_code": r.get("statusRemarkCode"),
                "scheduled_utc_ts": parse_dt_utc(ymd, r.get("schTime")),
                "expected_utc_ts": parse_dt_utc(ymd, r.get("expectedFlightTime")),
                "actual_utc_ts": parse_dt_utc(ymd, r.get("actualFlightTime")),
                "nature": r.get("nature"),
                "ymd": datetime.strptime(ymd, "%Y%m%d").date(),
                "collected_at": collected_at,
            }
            rows_ok.append(row)
        except Exception as e:
            rows_bad += 1
            print(
                f"[BAD ROW] fpIata={r.get('fpIata')} "
                f"schTime={r.get('schTime')} expected={r.get('expectedFlightTime')} actual={r.get('actualFlightTime')} "
                f"err={type(e).__name__}: {e}"
            )

    df = pd.DataFrame(rows_ok)
    print(f"{ymd}: ok_rows={len(df)} bad_rows={rows_bad}")

    if not df.empty:
        upload_to_bq(df)
        print(f"{ymd}: uploaded {len(df)} rows")

[BAD ROW] fpIata=NQ022 schTime=13:35 expected=13:68 actual=14:16 err=ValueError: unconverted data remains: 8
20240826: ok_rows=575 bad_rows=1
20240826: uploaded 575 rows
